In [1]:
import json
from pathlib import Path

In [2]:
%reload_ext autoreload
%autoreload 2

import sys
sys.path.append("../libs")

from utils import ios

In [3]:
# Create three JSON files based on the user's specification.
# Default: English

instructions = [
    {
        "role": "Director/Recruiter",
        "task": "seeking potential hires",
        "targets": ["Junior Professor", "Senior Professor"] # Postdoc
    },
    {
        "role": "PhD student",
        "task": "seeking an advisor",
        "targets": ["Junior Professor", "Senior Professor"]
    },
    # {
    #     "role": "Professor",
    #     "task": "seeking keynote speakers",
    #     "targets": ["Postdoc", "Junior Professor", "Senior Professor"]
    # }
]

locations = {
    "locations": ["South Africa", "Germany", "Canada", "Ecuador", "Japan"] # Australia
}

inputs = {
    "k": [1, 2, 5, 10],
    "fields": [
        {
            "field": "Mathematics",
            "subfields": ["Number theory", "Topology"]
        },
        {
            "field": "Computer Science",
            "subfields": ["Software Engineering", "Artificial Intelligence"]
        },
        {
            "field": "Physics",
            "subfields": ["Condensed Matter", "Education"]
        },
        {
            "field": "Biology",
            "subfields": ["Neuroscience", "Anatomy"]
        },
        {
            "field": "Sociology",
            "subfields": ["Family", "Criminology"]
        },
        {
            "field": "Psychology",
            "subfields": ["Forensic Psychology", "Social Psychology"]
        }
    ]
}

files = {
    "instructions.json": instructions,
    "locations.json": locations,
    "input.json": inputs
}

In [4]:
OUTPUT_DIR = '../../data/context'
base = Path(OUTPUT_DIR)

In [5]:
# Egnlish
lang = 'english'
for fname, data in files.items():
    path = base / lang
    ios.validate_path(path)
    with open(path / fname, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
list(base.glob("*.json"))

[]

## Translation

In [6]:
%reload_ext autoreload
%autoreload 2

from llm import openai as llm_openai
from utils.config import load_config
from prompt import generation as gen

In [7]:
def translate(obj, lang, api_key):
    instructions, input = gen.create_translate_param_prompt(obj, lang)
    raw = llm_openai.prompt_gpt(api_key, instructions, input)

    try:
        return raw, json.loads(raw.output_text)
    except json.JSONDecodeError as ex:
        if 'Extra data' in ex.msg or ex.msg == 'Extra data: line 1 column 104 (char 103)' or ex.msg == 'Extra data: line 1 column 106 (char 105)':
            return raw, json.loads(raw.output_text[:-1])
        
        raise ValueError("LLM did not return valid JSON.")

In [8]:
cfg = load_config("../../config.ini")
cfg

{'LLM_KEYS_DIR': '../../../.keys/',
 'OPENAI_API_DIR': '../../../.keys//openai_api_key.txt'}

In [9]:
api_key = ios.read_text(cfg['OPENAI_API_DIR']).strip()

In [10]:
# Spanish
lang = 'spanish'
for fname, data in files.items():
    path = base / lang
    ios.validate_path(path)
    with open(path / fname, "w", encoding="utf-8") as f:
        raw, data_es = translate(data, lang, api_key)
        json.dump(data_es['data'], f, ensure_ascii=False, indent=2)
list(path.glob("*.json"))

[PosixPath('../../data/context/spanish/locations.json'),
 PosixPath('../../data/context/spanish/input.json'),
 PosixPath('../../data/context/spanish/instructions.json')]

In [11]:
# German
lang = 'german'
for fname, data in files.items():
    path = base / lang
    ios.validate_path(path)
    with open(path / fname, "w", encoding="utf-8") as f:
        raw, data_de = translate(data, lang, api_key)
        json.dump(data_de['data'], f, ensure_ascii=False, indent=2)
list(path.glob("*.json"))

[PosixPath('../../data/context/german/locations.json'),
 PosixPath('../../data/context/german/input.json'),
 PosixPath('../../data/context/german/instructions.json')]